# PGA Query Engine
Core pipeline: Natural Language → SQL → Execute → Explain

Run cells top to bottom. Each section can be tested independently.

## 1. Imports & Environment

In [ ]:
import os
import json
import anthropic
import psycopg2
import psycopg2.extras
import nbformat
from dotenv import load_dotenv

load_dotenv()

# Verify credentials loaded
print("ANTHROPIC_API_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("DB_HOST:", os.getenv("DB_HOST"))
print("DB_NAME:", os.getenv("DB_NAME"))
print("DB_USER:", os.getenv("DB_USER"))
print("DB_PORT:", os.getenv("DB_PORT"))

In [ ]:
%run schema.ipynb

## 2. Load Schema Context
Run schema notebook first, or redefine here.

In [ ]:
# Option A: Run the schema notebook inline
# %run schema.ipynb

# Option B: Import from schema.py if you prefer .py files for schema
# from schema import NL_TO_SQL_SYSTEM, EXPLAIN_RESULTS_SYSTEM

# Option C: Redefine inline — copy NL_TO_SQL_SYSTEM and EXPLAIN_RESULTS_SYSTEM 
# from schema.ipynb if needed

# After running schema.ipynb, these variables should already be in memory:
print("NL_TO_SQL_SYSTEM defined:", 'NL_TO_SQL_SYSTEM' in dir())
print("EXPLAIN_RESULTS_SYSTEM defined:", 'EXPLAIN_RESULTS_SYSTEM' in dir())

## 3. Anthropic Client

In [ ]:
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
print("Anthropic client initialized")

## 4. Database Config

In [ ]:
db_config = {
    "host":     os.getenv("DB_HOST", "localhost"),
    "port":     int(os.getenv("DB_PORT", 5432)),
    "database": os.getenv("DB_NAME"),
    "user":     os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
}

print("DB config loaded:")
for k, v in db_config.items():
    print(f"  {k}: {'*****' if k == 'password' else v}")

## 5. Test Database Connection

In [ ]:
try:
    conn = psycopg2.connect(**db_config)
    print("✅ Database connection successful")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")

## 6. Step 1 — Generate SQL from Natural Language

In [ ]:
def generate_sql(question: str) -> str:
    """Convert a natural language question to a PostgreSQL query."""
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1000,
        system=NL_TO_SQL_SYSTEM,
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text.strip()

print("generate_sql() defined")

### Test SQL Generation
Change the question below and re-run to test different queries.

In [ ]:
test_question = "Who are the top 10 golfers by composite score?"

generated_sql = generate_sql(test_question)
print(f"Question: {test_question}")
print(f"\nGenerated SQL:\n{generated_sql}")

## 7. Step 2 — Execute SQL Query

In [ ]:
def execute_query(sql: str) -> tuple:
    """Run SQL and return (rows as list of dicts, column names)."""
    conn = psycopg2.connect(**db_config)
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql)
            rows = cur.fetchall()
            columns = [desc[0] for desc in cur.description]
            return [dict(row) for row in rows], columns
    finally:
        conn.close()

print("execute_query() defined")

### Test Query Execution
Uses the SQL generated in the cell above.

In [ ]:
try:
    rows, columns = execute_query(generated_sql)
    print(f"✅ Query returned {len(rows)} rows")
    print(f"Columns: {columns}")
    print(f"\nFirst 3 rows:")
    for row in rows[:3]:
        print(row)
except Exception as e:
    print(f"❌ Query failed: {e}")

## 8. Step 3 — Explain Results in Plain English

In [ ]:
def explain_results(question: str, sql: str, rows: list, columns: list) -> str:
    """Generate a plain English explanation of query results."""
    sample = rows[:20]
    results_text = json.dumps(sample, indent=2, default=str)

    prompt = f"""
Original question: {question}
SQL query used: {sql}
Results ({len(rows)} total rows):
{results_text}
Columns returned: {', '.join(columns)}
"""
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=500,
        system=EXPLAIN_RESULTS_SYSTEM,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("explain_results() defined")

### Test Explanation

In [ ]:
try:
    explanation = explain_results(test_question, generated_sql, rows, columns)
    print(f"Question: {test_question}")
    print(f"\nExplanation:\n{explanation}")
except Exception as e:
    print(f"❌ Explanation failed: {e}")

## 9. Full Pipeline Function

In [ ]:
def run_nl_query(question: str) -> dict:
    """
    Full pipeline: NL question → SQL → execute → explain.
    Returns dict with: question, sql, rows, columns, explanation, error
    """
    result = {
        "question": question,
        "sql": None,
        "rows": [],
        "columns": [],
        "explanation": None,
        "error": None
    }

    # Step 1: Generate SQL
    try:
        sql = generate_sql(question)
        if sql.startswith("ERROR:"):
            result["error"] = sql
            return result
        result["sql"] = sql
        print(f"✅ SQL generated")
    except Exception as e:
        result["error"] = f"Failed to generate SQL: {str(e)}"
        return result

    # Step 2: Execute
    try:
        rows, columns = execute_query(sql)
        result["rows"] = rows
        result["columns"] = columns
        print(f"✅ Query returned {len(rows)} rows")
    except Exception as e:
        result["error"] = f"Query execution failed: {str(e)}\n\nSQL:\n{sql}"
        return result

    # Step 3: Explain
    try:
        if rows:
            result["explanation"] = explain_results(question, sql, rows, columns)
        else:
            result["explanation"] = "No results found. Filters may be too narrow."
        print(f"✅ Explanation generated")
    except Exception as e:
        result["explanation"] = f"Results retrieved but explanation failed: {str(e)}"

    return result

print("run_nl_query() defined")

## 10. End-to-End Test
Change the question and run to test the full pipeline.

In [ ]:
question = "Who are the top 10 golfers by count of finishes in the top 20 in the year 2026?"

result = run_nl_query(question)

if result["error"]:
    print(f"❌ Error: {result['error']}")
else:
    print("=" * 60)
    print(f"QUESTION: {result['question']}")
    print("=" * 60)
    print(f"\nSQL:\n{result['sql']}")
    print(f"\nROWS RETURNED: {len(result['rows'])}")
    print(f"\nEXPLANATION:\n{result['explanation']}")
    print(f"\nFIRST 10 ROWS:")
    for row in result['rows'][:10]:
        print(row)

## 11. Batch Test — Multiple Questions
Run several questions at once to stress test SQL generation.

In [ ]:
test_questions = [
    "Which golfers have the best strokes gained tee-to-green in the last 10 rounds?",
    "Show me players with a cut rate above 75% last year",
    "Who has the most top-10 finishes in the last 5 events?",
    "Who won the most prize money in major tournaments?",
]

print("Testing SQL generation only (no DB execution):\n")
for q in test_questions:
    sql = generate_sql(q)
    print(f"Q: {q}")
    print(f"SQL: {sql}\n")
    print("-" * 60)